In [18]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000020E42796710>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020E42797390>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Fixed spelling: 'retrieval' instead of 'retreival'
from langchain.chains import create_retrieval_chain

# Fixed function name: 'create_stuff_documents_chain' instead of 'stuff_documents_chain'
from langchain.chains.combine_documents import create_stuff_documents_chain

In [8]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [9]:
import bs4
loader =WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs=loader.load()
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

e:\gen_ai\langchain\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7443.37it/s]


In [10]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
split_docs=text_splitter.split_documents(docs)
vectors=Chroma.from_documents(split_docs,embedding=embeddings)
retreiver=vectors.as_retriever()
retreiver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000020E30803B60>, search_kwargs={})

In [11]:
system_prompt = (
    "You are a helpful AI assistant for question answering. "
    "Use only the provided context to answer the user's question. "
    "If the answer is not present in the context, simply say "
    "'I don't know based on the provided context.' "
    "Do not make up or hallucinate information. "
    "Keep your answers clear, accurate, and concise.\n\n"
    "Context:\n{context}"
)

In [12]:
prompts=ChatPromptTemplate.from_messages([ 
("system",system_prompt),
("human","{input}")
 
])

In [19]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 1. Create the document chain (passing LLM first, prompt second)
question_ans_chain = create_stuff_documents_chain(llm, prompts)

# 2. Create the RAG chain (Fixed typo from 'retreiver' to 'retriever')
rag_chain = create_retrieval_chain(retreiver, question_ans_chain)

# 3. Invoke your chain
response = rag_chain.invoke({"input": "What is Artificial Intelligence?"})

# 4. Print the result
print(response["answer"])

Artificial Intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.


In [20]:
## adding chat history now


In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder 
from langchain_core.prompts import ChatPromptTemplate 

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"), 
        ("human", "{input}")
    ]
)

In [25]:
history_aware_retriever=create_history_aware_retriever(llm,retreiver,contextualize_q_prompt)

In [31]:
from langchain.chains.combine_documents import create_stuff_documents_chain

# 1. Define the system instructions for answering the question
qa_system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know.\n\n"
    "{context}"
)

# 2. Build the final chat prompt template
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

# 3. Create the document-stuffing chain
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [32]:
from langchain_core.messages import HumanMessage, AIMessage

# Simulate a conversation where we already talked about Artificial Intelligence
mock_chat_history = [
    HumanMessage(content="What is Artificial Intelligence?"),
    AIMessage(content="AI is the simulation of human intelligence processes by machines.")
]

In [33]:
response = rag_chain.invoke({
    "chat_history": mock_chat_history,
    "input": "Who coined this term?" # 'this term' relies on the history!
})

print("Answer:", response["answer"])

Answer: The term "Artificial Intelligence" was coined by John McCarthy.


In [36]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import ChatMessageHistory

# 1. Initialize the global dictionary store
store = {}

# 2. Define the history getter function
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

ImportError: cannot import name 'ChatMessageHistory' from 'langchain_core.chat_history' (e:\gen_ai\langchain\venv2\Lib\site-packages\langchain_core\chat_history.py)

In [39]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# The ultimate wrapper that handles multi-user/multi-session tracking automatically
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [41]:
# First turn of conversation
config = {"configurable": {"session_id": "user_session_123"}}

response1 = conversational_rag_chain.invoke(
    {"input": "What is ai?"},
    config=config
)
print("AI Response 1:", response1["answer"])

# Second turn (The chatbot will successfully remember what "it" means because of history tracking!)
response2 = conversational_rag_chain.invoke(
    {"input": "How do I install it?"},
    config=config
)
print("\nAI Response 2:", response2["answer"])

AI Response 1: According to the text, Artificial Intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

AI Response 2: I don't know how to install AI in general, as it's a broad field of research and development. The text provided doesn't include installation instructions for AI. AI can refer to a wide range of systems, software, and technologies, and the installation process would depend on the specific AI system or tool you are trying to install. If you could provide more context or specify which AI system or tool you are trying to install, I may be able to help 